# Imitation — permutación espectral compleja
Ordenamiento por magnitud sin sustituir coeficientes. Se mantienen DC y Nyquist. Las cuatro figuras se generan para cada secreto. En Colab sube Entradas_Originales.zip y ejecuta todas las celdas; descarga Paquete_Encoded_FFT.zip desde Archivos.

In [1]:
from pathlib import Path
import numpy as np
from scipy.io.wavfile import read, write
from scipy.fft import rfft, irfft, rfftfreq
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import zipfile
BASE = Path.cwd()
# En Colab, subir Entradas_Originales.zip o ejecutar desde la carpeta del repositorio.
if not (BASE/'Originales/target.wav').exists() and (BASE/'Entradas_Originales.zip').exists():
    with zipfile.ZipFile(BASE/'Entradas_Originales.zip') as z:
        for name in ['target.wav']+[f'secret{i}.wav' for i in range(1,6)]:
            (BASE/'Originales').mkdir(exist_ok=True)
            (BASE/'Originales'/name).write_bytes(z.read(name))
for folder in ['Codificados','Graficas','Resultados']:
    (BASE/folder).mkdir(exist_ok=True)
def cargar(ruta):
    fs, x = read(ruta)
    if x.ndim != 1: raise ValueError('Se requiere audio mono')
    if np.issubdtype(x.dtype,np.signedinteger):
        x=x.astype(float)/(-np.iinfo(x.dtype).min)
    elif x.dtype == np.uint8:
        x=(x.astype(float)-128)/128
    else: x=x.astype(float)
    if not np.isfinite(x).all(): raise ValueError('Muestras no finitas')
    return fs,x
fs, target = cargar(BASE/'Originales/target.wav')
N=len(target); T=rfft(target); f=rfftfreq(N,1/fs); t=np.arange(N)/fs
P=np.arange(1,len(T)-(N%2==0))
I2=P[np.argsort(np.abs(T[P]),kind='stable')]
print(f'Target: {fs} Hz, {N} muestras, {N/fs:.6f} s, {len(T)} coeficientes')
display(Audio(target,rate=fs))


Target: 44100 Hz, 383184 muestras, 8.688980 s, 191593 coeficientes


In [2]:
def figura(nombre, series, eje, unidad, titulo):
    fig, axs=plt.subplots(len(series),1,figsize=(10,2.3*len(series)),sharex=True,squeeze=False)
    for ax,(etiqueta,y) in zip(axs[:,0],series):
        ax.plot(eje,y,lw=.55); ax.set_ylabel(unidad); ax.set_title(etiqueta,loc='left',fontsize=10); ax.grid(alpha=.2)
    axs[-1,0].set_xlabel('Tiempo (s)' if unidad=='Amplitud' else 'Frecuencia (Hz)')
    fig.suptitle(titulo); fig.tight_layout(); fig.savefig(BASE/'Graficas'/nombre,dpi=125); plt.show(); plt.close(fig)

def db(X): return 20*np.log10(np.maximum(np.abs(X)/N,1e-12))
observaciones=[]
for i in range(1,6):
    rate, s=cargar(BASE/'Originales'/f'secret{i}.wav')
    if rate!=fs or len(s)!=N: raise ValueError(f'Secret {i}: Fs o N diferente')
    print(f'Secret {i}: {rate} Hz, {len(s)} muestras')
    display(Audio(s,rate=fs))
    S=rfft(s)
    I1=P[np.argsort(np.abs(S[P]),kind='stable')]
    K=np.arange(len(S)); K[I2]=I1
    E=S[K]
    assert np.array_equal(np.sort(K),np.arange(len(S)))
    assert K[0]==0 and (N%2!=0 or K[-1]==len(S)-1)
    assert np.array_equal(np.sort(np.abs(E)),np.sort(np.abs(S)))
    assert np.array_equal(np.argsort(np.abs(E[P]),kind='stable'),np.argsort(np.abs(T[P]),kind='stable')) or np.allclose(np.abs(E[I2]),np.sort(np.abs(S[P])))
    encoded=irfft(E,n=N)
    write(BASE/'Codificados'/f'encoded{i}.wav',fs,encoded.astype(np.float32))
    np.savetxt(BASE/'Codificados'/f'FFT_key{i}.csv',K,fmt='%d',delimiter=',',header='original_index',comments='')
    figura(f'{i}_tiempo.png',[('Target',target),('Secret',s)],t,'Amplitud',f'Mensaje {i} | señales originales')
    figura(f'{i}_espectro.png',[('Target',db(T)),('Secret',db(S))],f,'Magnitud (dB)',f'Mensaje {i} | espectros originales')
    figura(f'{i}_espectro_reorganizado.png',[('Target',db(T)),('Secret',db(S)),('Encoded',db(E))],f,'Magnitud (dB)',f'Mensaje {i} | redistribución por frecuencia')
    figura(f'{i}_encoded.png',[('Encoded',encoded)],t,'Amplitud',f'Mensaje {i} | señal codificada')
    display(Audio(encoded,rate=fs))
    def actividad(x):
        bloques=x[:len(x)//441*441].reshape(-1,441)
        return 100*np.mean(np.sqrt(np.mean(bloques**2,axis=1))>0.01)
    observaciones.append([i,np.max(abs(s)),np.max(abs(target)),actividad(s),actividad(target),actividad(encoded),f[P[np.argmax(abs(S[P]))]],f[P[np.argmax(abs(T[P]))]],f[P[np.argmax(abs(E[P]))]]])
    print(f'Clave {i}: {len(K)} índices válidos; magnitudes conservadas')
import csv
with (BASE/'Resultados/observaciones.csv').open('w',newline='') as h:
    w=csv.writer(h); w.writerow(['mensaje','pico_secret','pico_target','actividad_secret_pct','actividad_target_pct','actividad_encoded_pct','pico_secret_Hz','pico_target_Hz','pico_encoded_Hz']); w.writerows(observaciones)


Output hidden; open in https://colab.research.google.com to view.

In [3]:
with zipfile.ZipFile(BASE/'Paquete_Encoded_FFT.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted((BASE/'Codificados').glob('*')): z.write(p,p.name)
print('Paquete_Encoded_FFT.zip listo para Reverse.')


Paquete_Encoded_FFT.zip listo para Reverse.


In [4]:
from google.colab import files

files.download("Paquete_Encoded_FFT.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
from pathlib import Path

graficas = sorted(Path("Graficas").glob("*.png"))

print("Total de gráficas:", len(graficas))

for i, archivo in enumerate(graficas, 1):
    print(f"{i}. {archivo.name}")

Total de gráficas: 20
1. 1_encoded.png
2. 1_espectro.png
3. 1_espectro_reorganizado.png
4. 1_tiempo.png
5. 2_encoded.png
6. 2_espectro.png
7. 2_espectro_reorganizado.png
8. 2_tiempo.png
9. 3_encoded.png
10. 3_espectro.png
11. 3_espectro_reorganizado.png
12. 3_tiempo.png
13. 4_encoded.png
14. 4_espectro.png
15. 4_espectro_reorganizado.png
16. 4_tiempo.png
17. 5_encoded.png
18. 5_espectro.png
19. 5_espectro_reorganizado.png
20. 5_tiempo.png
